In [15]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

In [16]:
embed = nn.Embedding(10, 3)

In [17]:
df = pd.read_csv("../data/ml-latest-small/ratings.csv")
df = df.drop(columns=["timestamp"])

# Encode user and movie IDs to contiguous 0-based indices
user2idx = {u: i for i, u in enumerate(df["userId"].unique())}
movie2idx = {m: i for i, m in enumerate(df["movieId"].unique())}

df["user"] = df["userId"].map(user2idx)
df["movie"] = df["movieId"].map(movie2idx)

n_users = df["user"].nunique()
n_movies = df["movie"].nunique()
print(f"Users: {n_users}, Movies: {n_movies}, Ratings: {len(df)}")
print(f"Sparsity: {1 - len(df) / (n_users * n_movies):.2%}")
df.head()


Users: 200948, Movies: 84432, Ratings: 32000204
Sparsity: 99.81%


,userId,movieId,rating,user,movie
0,1,17,4.0,0,0
1,1,25,1.0,0,1
2,1,29,2.0,0,2
3,1,30,5.0,0,3
4,1,32,5.0,0,4


In [ ]:
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user"].values, dtype=torch.long)
        self.movies = torch.tensor(df["movie"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]


train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_loader = DataLoader(RatingsDataset(train_df), batch_size=1024, shuffle=True)
test_loader = DataLoader(RatingsDataset(test_df), batch_size=1024, shuffle=False)

class MF_MLP(nn.Module):
    def __init__(self, n_users: int, n_movies: int, emb_dim: int = 64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_movies, emb_dim)
        self.user_emb.weight.data.uniform_(0, 0.05)
        self.item_emb.weight.data.uniform_(0, 0.05)

    def forward(
        self, user_ids: torch.LongTensor, movie_ids: torch.LongTensor
    ) -> torch.Tensor:
        user_embeddings = self.user_emb(user_ids)
        item_embeddings = self.item_emb(movie_ids)
        return (user_embeddings * item_embeddings).sum(dim=1)


print("Creating model...")
model = MF_MLP(n_users, n_movies, emb_dim=64)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
print(model)
# print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


print("Training model...")
num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    for (
        user_batch,
        movie_batch,
        rating_batch,
    ) in train_loader:
        # print(f"Epoch {epoch:2d}  batch loss: ...", end="\r")
        # set gradients to zero
        optimizer.zero_grad()
        # make predictions
        y_hat = model(user_batch, movie_batch)
        # compute loss
        loss = loss_fn(y_hat, rating_batch)
        # compute gradients
        loss.backward()
        # update parameters
        optimizer.step()
        # accumulate loss for RMSE calculation
        total_loss += loss.item() * len(rating_batch)
        print(f"Epoch {epoch:2d}  batch loss: {loss.item():.4f}", end="\r")
    train_rmse = (total_loss / len(train_loader.dataset)) ** 0.5
    print(f"Epoch {epoch:2d}/{num_epochs}  train RMSE: {train_rmse:.4f}")


model.eval()
total_loss = 0
with torch.no_grad():
    for users, movies, ratings in test_loader:
        y_hat = model(users, movies)
        total_loss += loss_fn(y_hat, ratings).item() * len(ratings)
test_rmse = (total_loss / len(test_loader.dataset)) ** 0.5
print(f"Test RMSE: {test_rmse:.4f}")


# sample predictions vs ground truth
sample = test_df.sample(10, random_state=0)
users = torch.tensor(sample["user"].values, dtype=torch.long)
movies = torch.tensor(sample["movie"].values, dtype=torch.long)
with torch.no_grad():
    y_hat = model(users, movies).numpy()

print(f"\n{'Actual':>8}  {'Predicted':>9}")
for actual, pred in zip(sample["rating"].values, y_hat):
    print(f"{actual:8.1f}  {pred:9.4f}")


Creating model...
MF_MLP(
  (user_emb): Embedding(200948, 64)
  (item_emb): Embedding(84432, 64)
)
Training model...
